In [34]:
from dotenv import load_dotenv

load_dotenv()

True

In [35]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to the local MCP server running in api:mcp-dev
# The server is the FastMCP instance from app/main_mcp.py
client = MultiServerMCPClient(
    {
        "inkwell_mcp": {
            "transport": "stdio",
            "command": "python",
            "args": ["-m", "app.main_mcp"],
            "env": {
                "PYTHONPATH": "/Users/Ilia_Naryshkin/projects/inkwell/api",
                "INKWELL_ENV": "development",
            },
        }
    }
)

In [36]:
# Get tools and resources from the MCP server
tools = await client.get_tools()

# Get resources
resources = await client.get_resources("inkwell_mcp")

# Get prompts via the session interface
async with client.session("inkwell_mcp") as session:
    prompts_result = await session.list_prompts()

print("=== TOOLS ===")
for tool in tools:
    print(f"{tool.name}")

print("\n=== RESOURCES ===")
for resource in resources:
    print(f"{resource}")

print("\n=== PROMPTS ===")
print("Prompts in result:")
for prompt in prompts_result.prompts:
    print(f"{prompt.name}")

=== TOOLS ===
health_check
list_articles
get_article
create_article
save_article
delete_article

=== RESOURCES ===
metadata={'uri': AnyUrl('inkwell://article-schemas')} data='{\n  "ArticleMeta": {\n    "description": "Article metadata summary returned in list endpoints.",\n    "properties": {\n      "slug": {\n        "title": "Slug",\n        "type": "string"\n      },\n      "title": {\n        "title": "Title",\n        "type": "string"\n      },\n      "status": {\n        "enum": [\n          "draft",\n          "published"\n        ],\n        "title": "Status",\n        "type": "string"\n      },\n      "tags": {\n        "items": {\n          "type": "string"\n        },\n        "title": "Tags",\n        "type": "array"\n      }\n    },\n    "required": [\n      "slug",\n      "title",\n      "status",\n      "tags"\n    ],\n    "title": "ArticleMeta",\n    "type": "object"\n  },\n  "Article": {\n    "$defs": {\n      "ArticleMeta": {\n        "description": "Article metadata 

In [40]:
from langchain.agents import create_agent

agent = create_agent(
    model="claude-haiku-4-5",
    tools=tools,
)

print("Agent created successfully!")

Agent created successfully!


In [45]:
from langchain.messages import HumanMessage

# Test the agent with a query
response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Let me know if Inkwell is online")]}
)

In [46]:
for msg in response["messages"]:
    print(f"\n{msg.__class__.__name__}:")
    print(f"{msg.content}")


HumanMessage:
Let me know if Inkwell is online

AIMessage:
[{'text': "I'll check if the Inkwell MCP server is online for you.", 'type': 'text'}, {'id': 'toolu_017YvqVhbD3ArGAwrdbisu2z', 'caller': {'type': 'direct'}, 'input': {}, 'name': 'health_check', 'type': 'tool_use'}]

ToolMessage:
[{'type': 'text', 'text': '{\n  "status": "ok",\n  "message": "Inkwell MCP API server is up"\n}', 'id': 'lc_c66c1133-2041-4437-8bfa-8be58d044cba'}]

AIMessage:
Great! ✅ **Inkwell is online and running**. The MCP API server is up and operational.
